In [1]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Navigate to your project folder
# CHECK: Make sure 'ai-error-tutor' matches exactly what you named the folder in Drive
project_path = '/content/drive/MyDrive/ai-error-tutor'

try:
    os.chdir(project_path)
    print(f"✅ Moved to: {os.getcwd()}")

    # 3. Move into 'notebooks' so your relative paths (../data) work correctly
    os.chdir('notebooks')
    print(f"✅ Final working directory: {os.getcwd()}")
except FileNotFoundError:
    print("❌ Error: Could not find the folder. Check your Google Drive structure!")

Mounted at /content/drive
✅ Moved to: /content/drive/MyDrive/ai-error-tutor
✅ Final working directory: /content/drive/MyDrive/ai-error-tutor/notebooks


In [2]:
# Install dependencies
!pip install transformers datasets torch pandas scikit-learn rouge-score sentencepiece

import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from torch.utils.data import Dataset

# Check if GPU is available (Recommended for faster training)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=63e41020a2fa9c90f28cd843fd8f860aefce149099b406bd28e34e3ec006ef23
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
Using device: cuda


In [3]:
# OPTION 2: Load from CSV (If you upload your file to Colab files tab)
df = pd.read_csv('../data/error_dataset.csv')

# Split data
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

Training samples: 10023
Validation samples: 2506


In [4]:
class ErrorDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input=256, max_target=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_input = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Construct the input prompt
        input_text = f"explain error: {row['error_type']} | message: {row['raw_error']} | code: {row['code_context']}"
        target_text = f"{row['friendly_explanation']} FIX: {row['suggested_fix']}"

        # Tokenize Input
        inputs = self.tokenizer(
            input_text,
            max_length=self.max_input,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Tokenize Target (Label)
        targets = self.tokenizer(
            target_text,
            max_length=self.max_target,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze(),
            'labels': targets['input_ids'].squeeze()
        }

# Initialize Tokenizer
tokenizer = T5Tokenizer.from_pretrained('t5-small')

# Create Datasets (Now properly placed outside the class)
train_dataset = ErrorDataset(train_df, tokenizer)
val_dataset = ErrorDataset(val_df, tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

training_args = TrainingArguments(
    output_dir='../models/error_tutor_model',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    save_total_limit=3,                              # ← added: keeps only 3 checkpoints
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Start Training — automatically resumes from last checkpoint if one exists
import os
from transformers.trainer_utils import get_last_checkpoint

# Check if a checkpoint exists
last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    last_checkpoint = get_last_checkpoint(training_args.output_dir)

# Start training (will start fresh if last_checkpoint is None)
trainer.train(resume_from_checkpoint=last_checkpoint)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss
1,0.606859,0.403506
2,0.376338,0.234320
3,0.307622,0.192817
4,0.232546,0.178927
5,0.321531,0.174952


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=6265, training_loss=0.49892101295452923, metrics={'train_runtime': 856.2649, 'train_samples_per_second': 58.527, 'train_steps_per_second': 7.317, 'total_flos': 3391327190384640.0, 'train_loss': 0.49892101295452923, 'epoch': 5.0})

In [6]:
save_path = '../models/error_tutor_model'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

# Optional: Download to your local machine (uncomment if needed)
# from google.colab import files
# !zip -r error_tutor_model.zip ./error_tutor_model
# files.download('error_tutor_model.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ../models/error_tutor_model


In [19]:
def explain_error(error_type, message, code):
    # Prepare input text matching the training format
    input_text = f"explain error: {error_type} | message: {message} | code: {code}"

    # Move inputs to the same device as the model (GPU or CPU)
    inputs = tokenizer(input_text, return_tensors='pt', max_length=256, truncation=True)
    input_ids = inputs['input_ids'].to(model.device)

    # Generate prediction
    outputs = model.generate(
        input_ids=input_ids,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test Case
result = explain_error(
   "TypeError",
"unsupported operand type(s) for +: 'int' and 'str'",
"age = 20\nmessage = 'I am ' + age + ' years old'\nprint(message)"
)

print("--- AI Tutor Response ---")
print(result)

--- AI Tutor Response ---
You are trying to use the '+' operator between a number and a text string, which Python doesn't allow. FIX: Ensure both variables are the same type before the operation, or convert one (e.g., str(age)).


In [20]:
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Test on a subset of the validation data to save time (e.g., 200 random samples)
# Remove `.sample(...)` to run on the entire validation set
test_subset = val_df

rouge1_f1_scores = []
rougeL_f1_scores = []
exact_matches = 0

print(f"Evaluating model on {len(test_subset)} validation samples...\n")

for _, row in tqdm(test_subset.iterrows(), total=len(test_subset)):
    # 1. Get the expected Target text
    target = f"{row['friendly_explanation']} FIX: {row['suggested_fix']}"

    # 2. Get AI Prediction
    prediction = explain_error(row['error_type'], row['raw_error'], row['code_context'])

    # 3. Calculate Exact Match
    if prediction.strip().lower() == target.strip().lower():
        exact_matches += 1

    # 4. Calculate ROUGE metrics
    scores = scorer.score(target, prediction)
    rouge1_f1_scores.append(scores['rouge1'].fmeasure)
    rougeL_f1_scores.append(scores['rougeL'].fmeasure)

# Calculate averages
avg_rouge1 = sum(rouge1_f1_scores) / len(rouge1_f1_scores)
avg_rougeL = sum(rougeL_f1_scores) / len(rougeL_f1_scores)
exact_match_acc = exact_matches / len(test_subset)

print("\n" + "="*40)
print("📊 MODEL ACCURACY & EVALUATION METRICS")
print("="*40)
print(f"✅ ROUGE-1 F1 Score : {avg_rouge1:.4f} (Measures single-word overlap accuracy)")
print(f"✅ ROUGE-L F1 Score : {avg_rougeL:.4f} (Measures phrase sequence accuracy)")
print(f"✅ Exact Match Acc  : {exact_match_acc:.2%} (Percentage of 100% identical responses)")
print("="*40)

Evaluating model on 2506 validation samples...



  0%|          | 0/2506 [00:00<?, ?it/s]


📊 MODEL ACCURACY & EVALUATION METRICS
✅ ROUGE-1 F1 Score : 0.8135 (Measures single-word overlap accuracy)
✅ ROUGE-L F1 Score : 0.7931 (Measures phrase sequence accuracy)
✅ Exact Match Acc  : 32.72% (Percentage of 100% identical responses)
